# Detección de agua e inundación en Lima Sur con Sentinel-1 y U-Net

**Requisito:** ejecutar antes `preparacion_datos.ipynb`. Este notebook no lee datos crudos: todo sale de `Data/procesado/preparado/` (parches empaquetados, normalización, decisiones de calidad y escenas de prueba ya alineadas). Sus resultados quedan en `Data/procesado/resultados/`.

| Ruta | Datos | Uso |
|---|---|---|
| **A** | Kuro Siwo Chad, Harvey y Yibuti | entrenamiento; validación = bloques geográficos de 10 km de Harvey (~20 %) |
| **B** | Sentinel-1 Lima 2017 + EMSN038; Lima Sur 2023 + CONIDA | prueba y aplicación, **sin** reentrenar |
| extra | Sen1Floods11 Bolivia (15 parches manuales) | comparación con la literatura |

Se entrenan dos configuraciones:

* **`agua`** (la del informe): 2 canales (VV, VH durante el evento) → presencia de agua.
* **`cambio`**: 4 canales (VV, VH antes + VV, VH durante) → inundación nueva. Kuro Siwo trae imagen previa, y Bountos et al. (2024) muestran que apilar fechas es lo que mejor funciona. Además ataca el problema principal de Lima: la arena del desierto es oscura en radar **en ambas fechas**, así que no “cambia”.

> **Qué responde y qué no:** el modelo detecta *dónde hubo agua en superficie en el momento en que pasó el satélite*. No predice lluvia ni pronostica inundaciones futuras.

Todo se configura en `src/config.py`. Con la variable de entorno `PML_RAPIDO=1` ambos notebooks corren de punta a punta en pocos minutos (prueba de humo).

## 0. Entorno y datos preparados

In [ ]:
import sys, json, warnings
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import numpy as np, pandas as pd, torch, geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display
from src import config as C
from src import datos as D, entrenamiento as E, inferencia as I, preparacion as P

faltan = [r for r in ["indice.csv", "stats.json", "calidad.csv", "particion.json", "empaquetado/rutas.csv",
                      "pruebas/bolivia.npz", "pruebas/lima_2017.npz", "pruebas/lima_2023.npz", "pruebas/distritos_lima.gpkg"]
          if not (C.PREPARADO / r).exists()]
assert not faltan, f"falta ejecutar preparacion_datos.ipynb: {faltan}"

C.RESULTADOS.mkdir(parents=True, exist_ok=True)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
REENTRENAR = False        # True para reentrenar aunque ya exista el .pt
indice = pd.read_csv(C.PREPARADO / "indice.csv")
stats = json.load(open(C.PREPARADO / "stats.json"))
calidad = pd.read_csv(C.PREPARADO / "calidad.csv")
print(f"preparado: {C.PREPARADO}\nresultados: {C.RESULTADOS}\ndispositivo: {DISPOSITIVO}"
      + (f" ({torch.cuda.get_device_name(0)})" if DISPOSITIVO == "cuda" else ""))
print(f"\nindice.csv: {len(indice)} parches · columnas {list(indice.columns)}")
display(indice.head(3))
pd.DataFrame(stats).set_index("canales").round(3)

## 1. Qué datos usa cada modo
Decisión tomada en `preparacion_datos.ipynb` (sección 5): qué eventos sirven en cada modo y por qué.

In [ ]:
display(calidad[["dataset", "evento", "particion", "sirve_agua", "motivo_agua", "sirve_cambio", "motivo_cambio"]])
pd.concat({m: indice[indice[m]].groupby(["particion", "dataset"]).agg(parches=("ruta", "size"),
                                                                     agua_media=("frac_agua", "mean"))
           for m in C.MODOS}, axis=1).round(3)

In [ ]:
# un parche de cada fuente, tal como lo ve el entrenamiento
ejemplos = indice[(indice.frac_agua > 0.05) & indice.tiene_pre].groupby("dataset").head(1)
fig, ejes = plt.subplots(len(ejemplos), 6, figsize=(16, 2.9 * len(ejemplos)), squeeze=False)
for fila, e in zip(ejes, ejemplos.itertuples()):
    d = np.load(e.ruta)
    print(e.dataset, {k: (d[k].shape, str(d[k].dtype)) for k in d.files})
    img = d["img"].astype("float32")
    for ax, x, t in zip(fila, [img[0], img[1], img[2], img[3], d["lab"], d["val"]],
                        C.NOMBRES_CANALES + ["etiqueta", "válido"]):
        cm, lo, hi = ("gray", -25, 0) if t.startswith("V") else ("viridis", 0, 3)
        ax.imshow(x, cmap=cm, vmin=lo, vmax=hi)
        ax.set_title(f"{e.dataset}/{e.evento} · {t}", fontsize=7)
        ax.axis("off")
plt.tight_layout()

## 2. Entrenamiento
Para cada modo y cada encoder (ResNet34, VGG19, EfficientNet-B0, MobileNetV2): U-Net preentrenada en ImageNet, BCE + Dice sobre píxeles válidos, AdamW (lr 1e-4) con reducción en meseta, precisión mixta, lotes con la mitad de parches positivos, rotaciones de 90° y reflexiones, parada temprana por IoU de validación (máximo 10 épocas, paciencia 3).

In [ ]:
resultados, historiales = [], []
for modo in C.MODOS:
    train, val = E.filtrar(indice, modo)
    print(f"\n== modo {modo}: train {len(train)} parches ({train.positivo.sum()} positivos) · "
          f"val {len(val)} ({val.positivo.sum()} positivos)")
    for enc in C.ENCODERS:
        ruta_pt = C.RESULTADOS / "modelos" / f"unet_{modo}_{enc}.pt"
        ruta_h = C.RESULTADOS / "modelos" / f"historial_{modo}_{enc}.csv"
        if ruta_pt.exists() and ruta_h.exists() and not REENTRENAR:
            h = pd.read_csv(ruta_h)
            mejor = h.loc[h.IoU.idxmax()].drop(["lr", "perdida_train"]).to_dict()
            print(f"{modo} {enc}: ya entrenado (IoU val {mejor['IoU']:.3f})")
        else:
            mejor, h = E.entrenar(enc, modo, train, val, stats, DISPOSITIVO)
            h.to_csv(ruta_h, index=False)
        resultados.append(mejor)
        historiales.append(h)
historial = pd.concat(historiales, ignore_index=True)
historial.to_csv(C.RESULTADOS / "historial.csv", index=False)

## 3. Comparación y selección del encoder (solo con validación internacional)

In [ ]:
resultados = pd.DataFrame(resultados).sort_values(["modo", "IoU"], ascending=[True, False]).reset_index(drop=True)
resultados.to_csv(C.RESULTADOS / "resultados_val.csv", index=False)
MEJOR = resultados.groupby("modo").first().encoder.to_dict()
print("Mejor encoder por modo:", MEJOR)
resultados.round(4)

In [ ]:
fig, ejes = plt.subplots(len(C.MODOS), 2, figsize=(11, 3.3 * len(C.MODOS)), squeeze=False)
for fila, (modo, gm) in zip(ejes, historial.groupby("modo")):
    for enc, g in gm.groupby("encoder"):
        fila[0].plot(g.epoca, g.perdida_train, marker="o", ms=3, label=enc)
        fila[1].plot(g.epoca, g.IoU, marker="o", ms=3, label=enc)
    fila[0].set_title(f"{modo} · pérdida train"); fila[1].set_title(f"{modo} · IoU val")
    for ax in fila:
        ax.set_xlabel("época"); ax.legend(fontsize=7)
plt.tight_layout()

## 4. Predicciones del mejor modelo en validación

In [ ]:
modelos = {m: I.cargar_modelo(m, MEJOR[m], DISPOSITIVO) for m in C.MODOS}
for modo, modelo in modelos.items():
    _, val = E.filtrar(indice, modo)
    ds = E.Parches(val, modo, stats, aumentar=False)
    muestras = val[val.positivo].groupby("evento").head(1).head(5)
    fig, ejes = plt.subplots(len(muestras), 3, figsize=(8, 2.7 * len(muestras)), squeeze=False)
    for fila, (k, e) in zip(ejes, muestras.iterrows()):
        x, y, v = ds[k]
        with torch.no_grad():
            p = torch.sigmoid(modelo(x[None].to(DISPOSITIVO)))[0, 0].cpu().numpy()
        for ax, z, t in zip(fila, [x[-1], y[0], (p >= C.UMBRAL) * v[0].numpy()], ["VH durante (norm.)", "referencia", "predicción"]):
            ax.imshow(z, cmap="gray" if t.startswith("VH") else "Blues"); ax.axis("off")
            ax.set_title(f"{modo} · {e.evento} · {t}", fontsize=7)
    plt.tight_layout(); plt.show()

## 5. Comparación adicional: Sen1Floods11 Bolivia (15 parches con etiqueta manual)
Solo modo `agua` (Bolivia no trae imagen previa). Referencia publicada: IoU de agua total ≈ 0.41 (FCNN, Bonafilia et al., 2020), con otro protocolo — la comparación es orientativa.

In [ ]:
bol = np.load(C.PRUEBAS / "bolivia.npz")
display(P.describir_npz(C.PRUEBAS / "bolivia.npz"))
c_unet, c_otsu = np.zeros(4), np.zeros(4)
for post, ref in zip(bol["post"], bol["ref"]):
    prob, valido = I.probabilidad(modelos["agua"], "agua", stats, post, None, DISPOSITIVO)
    m = valido & (ref != -1)
    c_unet += I.conteos(prob[m] >= C.UMBRAL, ref[m] == 1)
    c_otsu += I.conteos(post[1][m] < I.otsu(post[1][m]), ref[m] == 1)
pd.DataFrame([{"metodo": f"U-Net agua {MEJOR['agua']}", **E.metricas(*c_unet)},
              {"metodo": "Otsu VH por parche", **E.metricas(*c_otsu)}]).round(3)

## 6. Lima 2017 frente a EMSN038 (prueba externa)

**Advertencia importante.** Las capas usadas como referencia, `Potential_Maximum_flood` (atributo `source`: *river* / *maximun height*) y `Riverbank`, describen la **zona de inundación potencial máxima** y las riberas de los ríos Rímac y Chillón: no son una extensión de agua observada el 17/03/2017. La celda siguiente comprueba que la retrodispersión dentro de esas zonas casi no cambia entre el 05/03 y el 17/03. Por eso las métricas de esta sección miden la coincidencia con una zona de peligro, no el acierto del modelo.

In [ ]:
capas17, perfil17 = P.cargar_escena(C.PRUEBAS / "lima_2017.npz")
display(P.describir_npz(C.PRUEBAS / "lima_2017.npz"))
P.mostrar_escena(C.PRUEBAS / "lima_2017.npz", "lima_2017.npz")
post17, pre17, ref17 = capas17["post"], capas17["pre"], capas17["ref"]
v17 = D.validos_db(post17) & D.validos_db(pre17) & (ref17 != -1)

def pct(x): return np.percentile(x, [10, 50, 90]).round(1).tolist()
pd.DataFrame([{"zona": n, "píxeles": m.sum(), "VH antes p10/50/90": pct(pre17[1][m]),
               "VH durante p10/50/90": pct(post17[1][m]), "ΔVH p10/50/90": pct((post17[1] - pre17[1])[m])}
              for n, m in [("EMSN038 (agua potencial)", v17 & (ref17 == 1)), ("resto del área", v17 & (ref17 == 0))]])

In [ ]:
hand17 = capas17["hand"].astype("float32")
bajo = hand17 < C.HAND_MAX
p_agua, _ = I.probabilidad(modelos["agua"], "agua", stats, post17, None, DISPOSITIVO)
p_cambio, _ = I.probabilidad(modelos["cambio"], "cambio", stats, post17, pre17, DISPOSITIVO)
u = I.otsu(post17[1][v17])
dvh = post17[1] - pre17[1]
capas17 = {f"U-Net agua {MEJOR['agua']}": p_agua >= C.UMBRAL,
           f"U-Net agua + HAND<{C.HAND_MAX}": (p_agua >= C.UMBRAL) & bajo,
           f"U-Net cambio {MEJOR['cambio']}": p_cambio >= C.UMBRAL,
           f"U-Net cambio + HAND<{C.HAND_MAX}": (p_cambio >= C.UMBRAL) & bajo,
           f"Otsu VH ({u:.1f} dB)": post17[1] < u,
           f"ΔVH < -3 dB + HAND<{C.HAND_MAX}": (dvh < -3) & bajo}
res17 = I.evaluar_mascaras(capas17, ref17, v17)
res17.round(3)

In [ ]:
I.mostrar([(np.ma.masked_where(~v17, post17[1]), "VH 17/03 (dB)", "gray", -25, 0),
           (np.ma.masked_where(~v17, dvh), "ΔVH (dB)", "RdBu", -6, 6),
           (np.ma.masked_where(~v17, p_agua), "prob. agua", "viridis", 0, 1),
           (np.ma.masked_where(~v17, p_cambio), "prob. cambio", "viridis", 0, 1),
           (np.ma.masked_where(ref17 == -1, ref17 == 1), "EMSN038 (potencial)", "Blues", 0, 1)], "Lima 2017")

## 7. Aplicación en Lima Sur 2023 (ciclón Yaku)
Imagen del 16/03/2023 (órbita 171 descendente) con la del 04/03/2023 como previa. Se exportan GeoTIFF de probabilidad y máscara para superponerlos en QGIS con los mapas CONIDA georreferenciados. La ocurrencia JRC > 80 % se marca como posible agua permanente (mar, lagunas) y no se cuenta como inundación.

In [ ]:
# libera las escenas de 2017 antes de cargar 2023
del post17, pre17, hand17, bajo, p_agua, p_cambio, dvh, capas17
import gc; gc.collect()

capas23, perfil23 = P.cargar_escena(C.PRUEBAS / "lima_2023.npz")
display(P.describir_npz(C.PRUEBAS / "lima_2023.npz"))
P.mostrar_escena(C.PRUEBAS / "lima_2023.npz", "lima_2023.npz")
post23, pre23 = capas23["post"], capas23["pre"]
hand23, jrc23 = capas23["hand"].astype("float32"), capas23["jrc"].astype("float32")
permanente = np.nan_to_num(jrc23) > 80
p23_agua, v23_agua = I.probabilidad(modelos["agua"], "agua", stats, post23, None, DISPOSITIVO)
p23_cambio, v23 = I.probabilidad(modelos["cambio"], "cambio", stats, post23, pre23, DISPOSITIVO)
filtro = (hand23 < C.HAND_MAX) & ~permanente
inund23 = (p23_cambio >= C.UMBRAL) & filtro & v23
agua23 = (p23_agua >= C.UMBRAL) & filtro & v23_agua
oscuro_ambas = (post23[1] < -20) & (pre23[1] < -20) & v23

carpeta = C.RESULTADOS / "predicciones"
I.guardar_tif(carpeta / "lima_2023_0316_prob_agua.tif", np.where(v23_agua, p23_agua, -1), perfil23, "float32", -1)
I.guardar_tif(carpeta / "lima_2023_0316_prob_cambio.tif", np.where(v23, p23_cambio, -1), perfil23, "float32", -1)
I.guardar_tif(carpeta / "lima_2023_0316_inundacion.tif", np.where(v23, inund23, 255), perfil23, "uint8", 255)
I.guardar_tif(carpeta / "lima_2023_0316_agua.tif", np.where(v23_agua, agua23, 255), perfil23, "uint8", 255)
print(f"agua (modo agua): {100 * agua23[v23_agua].mean():.2f} % · inundación nueva (modo cambio): {100 * inund23[v23].mean():.2f} % · "
      f"oscuro en ambas fechas (arena/sombra): {100 * oscuro_ambas[v23].mean():.1f} %")
I.mostrar([(np.ma.masked_where(~v23, pre23[1]), "VH 04/03", "gray", -25, 0),
           (np.ma.masked_where(~v23, post23[1]), "VH 16/03", "gray", -25, 0),
           (np.ma.masked_where(~v23_agua, agua23), "agua (2 canales)", "Blues", 0, 1),
           (np.ma.masked_where(~v23, inund23), "inundación nueva (4 canales)", "Blues", 0, 1)], "Lima Sur 2023")

### 7.1 ¿En qué distritos? Área detectada por distrito (provincia de Lima)

In [ ]:
distritos = gpd.read_file(C.PRUEBAS / "distritos_lima.gpkg")
por_distrito = I.resumen_por_distrito(inund23, v23, perfil23, distritos).merge(
    I.resumen_por_distrito(agua23, v23_agua, perfil23, distritos)[["distrito", "km2_detectados"]]
     .rename(columns={"km2_detectados": "km2_agua_2canales"}), on="distrito", how="left")
por_distrito.to_csv(C.RESULTADOS / "lima_2023_por_distrito.csv", index=False)
por_distrito.round(3).head(20)

## 8. Lectura de resultados y limitaciones

* **Calidad de las etiquetas:** solo se entrena con Kuro Siwo (etiqueta a nivel de píxel). Se descartaron Sen1Floods11 Colombia (umbral Otsu sobre la propia imagen), UrbanSARFloods (polígonos gruesos y ríos sin etiquetar) y Kuro Siwo Madagascar (casi todo ríos y cuerpos > 100 ha); ver `preparacion_datos.ipynb`.
* **Manchas chicas:** aun en Kuro Siwo, solo el 2–5 % del agua está en manchas < 1 ha. A 10 m con speckle, un charco de 20×20 m son 4 píxeles: está en el límite de lo que Sentinel-1 puede ver.
* Validación internacional alta + prueba en Lima baja = **cambio de dominio**, no necesariamente un mal modelo. Hay que reportarlo así.
* **EMSN038 no es verdad de campo** para el 17/03/2017 (zona potencial + riberas). Recomendación: presentarla como “coincidencia con zonas de peligro” o reemplazarla por polígonos digitalizados de eventos observados (CONIDA, INDECI/SINPAD, notas de prensa con coordenadas).
* **Desierto costero:** buena parte de Lima Sur aparece oscura en VH en *ambas* fechas (arena seca, sombras de radar en laderas). El modelo de 2 canales lo confunde con agua; el de 4 canales lo ve como “sin cambio”.
* **Escala temporal:** los anegamientos y huaicos de Lima Sur duran horas y dejan lodo (que no es oscuro en radar). Sentinel-1 pasa cada 12 días a las ~05:40 h; si la lluvia no coincide con la pasada, el agua ya no está.
* **Resolución:** 10 m es grueso para calles inundadas; la inundación urbana real produce doble rebote (más brillo), que solo se separa bien con coherencia SLC, no con GRD.
* Para **predecir** zonas propensas (no detectar), se necesitaría otro planteamiento: precipitación (SENAMHI, GPM IMERG, CHIRPS) + susceptibilidad (DEM, pendiente, HAND, quebradas) + inventario histórico de eventos (SIGRID/CENEPRED, INGEMMET) como etiqueta.